In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os

# =============================================
# 1. Set your data path (adjust if needed)
# =============================================
data_path = r"C:\Users\NARKEES SALEEM\Downloads\store-sales-time-series-forecasting (1)"

# =============================================
# 2. Load data (use a sample for speed)
# =============================================
print("Loading data...")
train = pd.read_csv(f"{data_path}/train.csv", parse_dates=['date'], nrows=500000)
train = train.drop('id', axis=1)

stores = pd.read_csv(f"{data_path}/stores.csv")
oil = pd.read_csv(f"{data_path}/oil.csv", parse_dates=['date'])
holidays = pd.read_csv(f"{data_path}/holidays_events.csv", parse_dates=['date'])

# =============================================
# 3. Feature engineering (lightweight version)
# =============================================
# Merge stores
train = train.merge(stores, on='store_nbr', how='left')

# Merge oil (forward fill with .ffill() – no warning)
oil = oil.sort_values('date')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
train = train.merge(oil, on='date', how='left')

# Holiday flag
holiday_dates = set(holidays['date'].dt.date)
train['is_holiday'] = train['date'].dt.date.isin(holiday_dates).astype(int)

# Time features
train['dayofweek'] = train['date'].dt.dayofweek
train['month'] = train['date'].dt.month
train['year'] = train['date'].dt.year

# Lags (simple, fast)
train = train.sort_values(['store_nbr', 'family', 'date'])
train['lag_1'] = train.groupby(['store_nbr', 'family'])['sales'].shift(1)
train['lag_7'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)

# Drop rows with NaN lags (first week per group)
train = train.dropna(subset=['lag_1', 'lag_7'])

# Encode family
le = LabelEncoder()
train['family_encoded'] = le.fit_transform(train['family'])

# Features to use
features = ['store_nbr', 'family_encoded', 'onpromotion', 'dcoilwtico',
            'is_holiday', 'dayofweek', 'month', 'year', 'lag_1', 'lag_7']

X = train[features]
y = train['sales']

print(f"Training on {len(X)} rows")

# =============================================
# 4. Train XGBoost
# =============================================
print("Training XGBoost...")
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
model.fit(X, y)

# Optional: evaluate on a validation split
split = int(0.8 * len(X))
X_val, y_val = X[split:], y[split:]
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
print(f"Validation RMSE: {rmse:.2f}, MAE: {mae:.2f}")

# =============================================
# 5. SAVE MODEL FILES IN THE CORRECT FOLDER
# =============================================
# Current working directory (should be FUTURE_ML_01)
save_dir = os.getcwd()
print(f"Saving model to: {save_dir}")

joblib.dump(model, "xgboost_model.pkl")
joblib.dump(le, "label_encoder.pkl")

# Save feature names for the dashboard
with open("feature_names.txt", "w") as f:
    f.write(",".join(features))

print("✅ Model files saved successfully:")
print("   - xgboost_model.pkl")
print("   - label_encoder.pkl")
print("   - feature_names.txt")

# Verify files exist
files = os.listdir(".")
for f in ["xgboost_model.pkl", "label_encoder.pkl", "feature_names.txt"]:
    if f in files:
        print(f"   ✓ {f} (size: {os.path.getsize(f)/1024:.1f} KB)")
    else:
        print(f"   ✗ {f} MISSING")

Loading data...
Training on 487526 rows
Training XGBoost...
Validation RMSE: 163.31, MAE: 40.52
Saving model to: C:\Users\NARKEES SALEEM
✅ Model files saved successfully:
   - xgboost_model.pkl
   - label_encoder.pkl
   - feature_names.txt
   ✓ xgboost_model.pkl (size: 862.1 KB)
   ✓ label_encoder.pkl (size: 0.9 KB)
   ✓ feature_names.txt (size: 0.1 KB)
